# 03_retail_demand_forecasting — Forecasting Model Comparison

Build time-series features, compare a baseline vs RandomForest, and evaluate on a time-based holdout.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import joblib, json

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'daily_sales.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head()


## Feature engineering (lags + rolling means + calendar)

In [ ]:

for lag in [1,7,14]:
    df[f'lag_{lag}'] = df['units_sold'].shift(lag)
df['roll_mean_7'] = df['units_sold'].shift(1).rolling(7).mean()
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

dff = df.dropna().reset_index(drop=True)
X = dff.drop(columns=['units_sold','date'])
y = dff['units_sold'].astype(float)

split = int(len(dff)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


## Baseline: last week's same day (lag_7)

In [ ]:

baseline = X_test['lag_7'].values
mae_b = mean_absolute_error(y_test, baseline)
mape_b = mean_absolute_percentage_error(y_test, baseline)
{'baseline_mae': mae_b, 'baseline_mape': mape_b}


## Model: RandomForestRegressor

In [ ]:

rf = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mape = mean_absolute_percentage_error(y_test, pred)
{'rf_mae': mae, 'rf_mape': mape}


## Plot actual vs predicted

In [ ]:

plt.figure(figsize=(10,4))
plt.plot(dff['date'].iloc[split:], y_test.values, label='Actual')
plt.plot(dff['date'].iloc[split:], pred, label='RF Predicted')
plt.title('Actual vs RF Forecast (Holdout)')
plt.xlabel('Date'); plt.ylabel('Units')
plt.legend()
plt.show()


## Save artifacts

In [ ]:

(ROOT/'models').mkdir(exist_ok=True)
(ROOT/'reports').mkdir(exist_ok=True)

joblib.dump(rf, ROOT/'models'/'rf_forecast_notebook.joblib')
Path(ROOT/'reports'/'model_comparison_notebook.json').write_text(json.dumps({
    'baseline_mae': float(mae_b), 'baseline_mape': float(mape_b),
    'rf_mae': float(mae), 'rf_mape': float(mape)
}, indent=2))
print('Saved.')
